In this project, we will be working on extracting text from images. After extracting the text we will apply some basic functions of OpenCV on that text to enhance it and to get more accurate results. This project will be very useful as it will save time and effort of typing from an image.

In [ ]:
#import requests to install tesseract
import requests

We will importing request library for fetching the url for git files and images.


In [ ]:
# Downloading tesseract-ocr file
r = requests.get("https://raw.githubusercontent.com/tesseract-ocr/tessdata/4.00/ind.traineddata", stream = True)

# Writing data to file to avoid path isuues
with open("/usr/share/tesseract-ocr/4.00/tessdata/ind.traineddata", "wb") as file:
    for block in r.iter_content(chunk_size = 1024):
         if block:
             file.write(block)

We will now download tesseract which is required for pytesseract library to run and save the file at the path in open() function.

In [ ]:
# Installing libraries required for optical character recognition
! apt install tesseract-ocr libtesseract-dev libmagickwand-dev

# Importing IPython to clear output which is not important
from IPython.display import HTML, clear_output
clear_output()

In this step we will install the required libraries for ocr and we will also import IPython fucntions to clear the undesired.

In [ ]:
# Installing pytesseract and opencv
! pip install pytesseract wand opencv-python
clear_output()

Now we will install pytesseract and opencv libraries.

In [ ]:
# Import libraries
from PIL import Image
import pytesseract
import cv2
import numpy as np
from pytesseract import Output
import re

In [ ]:
from google.colab import files
from PIL import Image
import io

uploaded = files.upload()

filename = next(iter(uploaded))

image = Image.open(io.BytesIO(uploaded[filename]))
image = image.resize((300, 150))

image.save("sample.png")

display(image)

## Importing required libraries

In this step, an image is uploaded directly to Google Colab, resized, saved as `sample.png`, and displayed for further OCR processing.

In [ ]:
# Simply extracting text from image
custom_config = r'-l eng --oem 3 --psm 6'
text = pytesseract.image_to_string(image,config=custom_config)
print(text)

Here we will be extracting the text from image with custom configuration.

In [ ]:
# Extracting text from image and removing irrelevant symbols from characters
try:
  text=pytesseract.image_to_string(image,lang="eng")
  characters_to_remove = "!()@—*“>+-/,'|£#%$&^_~"
  new_string = text
  for character in characters_to_remove:
    new_string = new_string.replace(character, "")
  print(new_string)
except IOError as e:
    print("Error (%s)." % e)

Now we will remove unwanted symbols from the text we extracted by replacing the symbol with an empty string.

In [ ]:
# Now we will perform opencv operations to get text from complex images
image = cv2.imread('sample.png')

The saved image is loaded with OpenCV using `cv2.imread()` so that image preprocessing operations can be applied before OCR.

In [ ]:
# get grayscale image
def get_grayscale(image):
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
gray = get_grayscale(image)
Image.fromarray(gray)

The image is converted to grayscale to reduce the amount of color information that must be processed. A grayscale image stores intensity values from 0 (black) to 255 (white). The `cv2.cvtColor()` method is used to convert the image from BGR color space to grayscale.

In [ ]:
# Noise removal
def remove_noise(image):
    return cv2.medianBlur(image, 5)

noise_removed = remove_noise(gray)
Image.fromarray(noise_removed)

Now we will blur the image so that we can remove the noise from the image. Here, the function cv2.medianBlur() computes the median of all the pixels under the kernel window and the central pixel is replaced with this median value. This is highly effective in removing noise.

In [ ]:
#thresholding
def thresholding(image):
    return cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
thresh = thresholding(gray)
Image.fromarray(thresh)

We will perform threshold transformation here.
cv2. If pixel value is greater than a threshold value, it is assigned one value (may be white), else it is assigned another value (may be black). The function used is cv2.threshold. First argument is the source image, which should be a grayscale image.


In [ ]:
# Erosion
def erode_image(image):
    kernel = np.ones((5, 5), np.uint8)
    return cv2.erode(image, kernel, iterations=1)

eroded = erode_image(gray)
Image.fromarray(eroded)

Here we are doing erode transformation.
cv2.erode() method is used to perform erosion on the image. The basic idea of erosion is just like soil erosion only, it erodes away the boundaries of foreground object (Always try to keep foreground in white). It is normally performed on binary images. It needs two inputs, one is our original image, second one is called structuring element or kernel which decides the nature of operation.

In [ ]:
# Morphological opening
def opening(image):
    kernel = np.ones((5, 5), np.uint8)
    return cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel)

opened = opening(gray)
Image.fromarray(opened)

Here we will perform morphological trandformation. It is useful in opening small holes inside the foreground objects, or small white points on the object.

In [ ]:
# Canny edge detection
def canny(image):
    return cv2.Canny(image, 100, 200)

edges = canny(gray)
Image.fromarray(edges)

Canny edge detection is used to identify strong intensity changes that correspond to edges in the image.

In [ ]:
#skew correction
def deskew(image):
    coords = np.column_stack(np.where(image > 0))
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return rotated
rotated = deskew(gray)
Image.fromarray(rotated)

Now we will apply deskewing on the image. Deskewing is a process whereby skew is removed by rotating an image by the same amount as its skew but in the opposite direction. This results in a horizontally and vertically aligned image where the text runs across the page rather than at an angle.

In [ ]:
#template matching
def match_template(image, template):
    return cv2.matchTemplate(image, template, cv2.TM_CCOEFF_NORMED)
match = match_template(gray, gray)
match

Here we are trying to match the image. As we are passing same image for matching we got the similarity of 99.99%. Here, template matching is a method for searching and finding the location of a template image in a larger image. OpenCV comes with a function cv2.matchTemplate() for this purpose. It simply slides the template image over the input image (as in 2D convolution) and compares the template and patch of input image under the template image.  

In [ ]:
# Drawing rectangle around text
img = cv2.imread('sample.png')
h, w, c = img.shape
boxes = pytesseract.image_to_boxes(img)
for b in boxes.splitlines():
    b = b.split(' ')
    img = cv2.rectangle(img, (int(b[1]), h - int(b[2])), (int(b[3]), h - int(b[4])), (0, 255, 0), 2)
Image.fromarray(img)

Now we will segregate every character in the text by creating a rectangle around it.

In [ ]:
# Drawing a rectangle around a specific pattern or word
img = cv2.imread('sample.png')
data = pytesseract.image_to_data(img, output_type=Output.DICT)

target_pattern = 'artificially'
n_boxes = len(data['text'])

for i in range(n_boxes):
    if int(data['conf'][i]) > 60:
        if re.match(target_pattern, data['text'][i], re.IGNORECASE):
            x, y, w, h = (data['left'][i], data['top'][i], data['width'][i], data['height'][i])
            img = cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)

Image.fromarray(img)

Similarly we can draw rectangle around the specific pattern or word.

# Conclusion:

I started by installing Tesseract, which is used for text extraction from images. After that, I worked on extracting text from an image using Tesseract. I also explored various OpenCV image transformation techniques and found that applying the right preprocessing steps is essential for accurately extracting text from complex images.

## Scope:

Different organization can use this to extract useful information from the image and store it. Individuals can use it for saving their time and effort for typing.